# Offline Langfuse Trace Evaluation with Ragas

This notebook demonstrates how to retrieve and evaluate Langfuse traces using [Ragas metrics](https://docs.ragas.io/en/latest/concepts/metrics/overview/). This technique is useful for evaluating RAG pipelines AFTER the fact, using actual traces and spans captured by Langfuse in production or staging environments.  

The scoring information is associated with traces and stored back in Langfuse, allowing you to further analyze results using [Langfuse dashboards](https://langfuse.com/docs/metrics/features/custom-dashboards) and other capabilities.

There are a variety of different [Ragas metrics](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/) available, offering both **reference-free** metrics (like Faithfulness and AnswerRelevancy) that don't require ground-truth data, and **reference-based** metrics (like ContextPrecision) that evaluate against a known correct answer.

## The Pre-Requisites
Before you using this notebook:
- understand the [core concepts](https://langfuse.com/docs/evaluation/core-concepts) of evaluations in Langfuse
- understand the [Ragas metrics](https://docs.ragas.io/en/latest/concepts/metrics/overview/) and their data requirements
- perform the pre-requisites steps outlined in the [README](../README.md).

## The Environment
Establish environment variables and initialize connections as applicable. 

**Recommended:** Take a copy of the `.env.example` (at the root of this repo), save as  `.env`, and update placeholder values in that file.  If you don't do this, you will be prompted to enter the required values during notebook execution.

In [ ]:
from __future__ import annotations

import getpass
import os

from dotenv import load_dotenv

# Load environment variables from .env file (if present)
load_dotenv()


def get_env_or_prompt(var_name: str, prompt_text: str, is_secret: bool = True) -> str:
    """Get environment variable or prompt the user for input.

    Args:
        var_name: Name of the environment variable.
        prompt_text: Text to display when prompting the user.
        is_secret: If True, use getpass to hide input (for API keys).

    Returns:
        The environment variable value or user-provided input.
    """
    value = os.getenv(var_name)
    if value:
        return value
    if is_secret:
        return getpass.getpass(prompt_text)
    return input(prompt_text)


# Langfuse configuration - get from env or prompt
os.environ["LANGFUSE_PUBLIC_KEY"] = get_env_or_prompt(
    "LANGFUSE_PUBLIC_KEY", "Enter Langfuse Public Key: "
)
os.environ["LANGFUSE_SECRET_KEY"] = get_env_or_prompt(
    "LANGFUSE_SECRET_KEY", "Enter Langfuse Secret Key: "
)
os.environ["LANGFUSE_HOST"] = get_env_or_prompt(
    "LANGFUSE_HOST",
    "Enter Langfuse Host URL (default: https://cloud.langfuse.com): ",
    is_secret=False,
) or "https://cloud.langfuse.com"

# OpenAI configuration
os.environ["OPENAI_API_KEY"] = get_env_or_prompt(
    "OPENAI_API_KEY", "Enter OpenAI API Key: "
)

Test Langfuse connection. You should see a success message if the connection is working.

In [ ]:
from langfuse import get_client

# Verify Langfuse connection
langfuse = get_client()
if langfuse.auth_check():
    print("✅ Langfuse client is authenticated and ready!")
else:
    print("❌ Authentication failed. Please check your credentials and host.")

Initialize Langfuse callback handler.  This will be used by the "task" application, written in LangChain, that generates the traces for evaluation.

In [ ]:
from langfuse.langchain import CallbackHandler  # LangFuse handler

langfuse_handler = CallbackHandler()

## The Dataset

For this example, we use the [FIQA (Financial Opinion Mining and Question Answering)](https://huggingface.co/datasets/explodinggradients/fiqa) dataset from Hugging Face. This dataset was prepared by the RAGAS team and contains pre-computed RAG pipeline outputs, making it ideal for demonstrating evaluation workflows without needing a live retrieval system.

The dataset contains the following columns:
- `question`: list[str] - These are the questions your RAG pipeline will be evaluated on.
- `answer`: list[str] - The answer generated from the RAG pipeline and given to the user.
- `contexts`: list[list[str]] - The contexts which were passed into the LLM to answer the question.
- `ground_truths`: list[list[str]] - The ground truth answer to the questions. However, this can be ignored for online evaluations since we will not have access to ground-truth data in our case.

See the [Hugging Face dataset page](https://huggingface.co/datasets/explodinggradients/fiqa) for more details on the dataset structure and provenance.

In [ ]:
from datasets import load_dataset

# Load FIQA dataset directly
fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval")["baseline"].select(
    range(1),
)  # limit to 1 items for testing; remove .select() for full dataset
fiqa_eval

## The LLM Clients
Define the client objects that will communicate with the LLMs.

In [ ]:
from langchain_openai import ChatOpenAI

# Initialize LLM client for Langchain task application
langchain_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)


In [ ]:
from openai import AsyncOpenAI
from ragas.embeddings.base import embedding_factory
from ragas.llms import llm_factory

# Initialize LLM client for RAGAS Metrics (the LangChain LLMWrapper was deprecated in RAGAS v0.4+)
async_client = AsyncOpenAI()
llm = llm_factory(
    "gpt-4o-mini", 
    client=async_client,
    max_tokens=8192,
)
embeddings = embedding_factory("openai", model="text-embedding-ada-002", client=async_client)

## The Ragas Metrics

Depending on what you are trying to optimize, you may want to use different metrics, or even custom metrics. This example uses the following Ragase metrics:

1. [Faithfulness](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/faithfulness/): This measures the factual consistency of the generated answer against the given context.
2. [AnswerRelevancy](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/answer_relevance/): Answer Relevancy, focuses on assessing how pertinent the generated answer is to the given prompt.
3. [ContextPrecision](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/context_precision/): Context Precision is a metric that evaluates whether all of the ground-truth relevant items present in the contexts are ranked high. Ideally all the relevant chunks must appear at the top ranks. This metric is computed using the question and the contexts, with values ranging between 0 and 1, where higher scores indicate better precision.

Checkout the [List of available metrics](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/) to know more about these metrics, how they work, and other metrics available for evaluation.

In [ ]:
from ragas.metrics.collections import AnswerRelevancy, ContextPrecision, Faithfulness

# Initialize metrics with the LLM
metrics = [
    Faithfulness(llm=llm),
    AnswerRelevancy(llm=llm, embeddings=embeddings, strictness=3),
    ContextPrecision(llm=llm),
]

## The Traces
This is a simple LangChain task that takes in the questions and contexts from the dataset, and queries the LLM for answers.  The Langfuse traces generated during this process will be used for evaluation in subsequent sections.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# define template that we will "stuff" with questions and context during task execution
prompt = ChatPromptTemplate.from_template("""
Answer based ONLY on the following contexts:
{contexts}

Question: {question}
""")

In [ ]:
from datetime import datetime
from typing import Any, Dict

run_name = "fiqa-eval" # define run name for LangFuse tracing
session_id = f"notebook: {datetime.now().isoformat()}" # define unique session id for LangFuse tracing

# execute chain for each item in the dataset, passing LangFuse callback handler in config to capture trace, and including ground truths in metadata for later analysis
for row in fiqa_eval:
    row_dict: Dict[str, Any] = dict(row)
    # extract contexts, question, and ground truths from the dataset row
    contexts_text = "\n".join(row_dict["contexts"])
    question_text = row_dict["question"]
    ground_truths_text = (
        "\n".join(row_dict["ground_truths"]) if row_dict.get("ground_truths") else None
    )
    # Build chain: prompt → LLM
    chain = prompt | langchain_llm

    # Invoke chain, injecting context and question directly, and pass LangFuse callback handler in config to capture trace
    result = chain.invoke(
        {"contexts": contexts_text, "question": question_text},
        config={
            "callbacks": [langfuse_handler],
            "run_name": run_name,
            "metadata": {
                "ground_truths": ground_truths_text, # include ground truths in metadata for later analysis
                "langfuse_session_id": session_id, # include session id in metadata to link traces together
            },
        },
    )

# Flush to ensure all traces are sent to Langfuse
langfuse.flush()

## The Evaluation
Take the traces created in the previous section and run Ragas evaluation on them. The results will be sent to Langfuse as trace scores, and you can also print them out in the notebook.

In [ ]:
import time

# Pause to ensure the Langfuse traces get processed by the server before proceeding to the next step
time.sleep(120)

Use Langfuse API to retrieve the trace for this run.

In [ ]:
# Retrieve all traces for this session
traces = langfuse.api.trace.list(session_id=session_id)
traces

Loop through the traces to collect contexts, question, answers, and ground truths (if applicable) for each trace.

In [ ]:
samples = []
trace_ids = []

# Collect samples from traces
for trace in traces.data:
    if trace.input and "contexts" in trace.input and "question" in trace.input:
        contexts = trace.input["contexts"]
        question = trace.input["question"]
    else:
        print(f"Trace ID: {trace.id} doesn't have 'contexts' and 'question' in input!")
        continue

    if trace.output and "content" in trace.output:
        answer = trace.output["content"]
    else:
        print(f"Trace ID: {trace.id} doesn't have content in output!")
        continue

    # Get ground_truths from metadata for ContextPrecision metric
    ground_truths = None
    if trace.metadata and "ground_truths" in trace.metadata:
        ground_truths = trace.metadata["ground_truths"]

    samples.append({
        "user_input": question,
        "response": answer,
        "retrieved_contexts": contexts.split("\n"),
        "reference": ground_truths,  # Required for ContextPrecision
    })
    trace_ids.append(trace.id)



Define helper methods to simplify the execution of the evaluation metrics.  This will include formatting the input data into the format required by the Ragas metrics, and executing the metric to get a score.

In [ ]:
import inspect

from ragas.metrics.collections.base import BaseMetric
from ragas.metrics.result import MetricResult


def get_ascore_params(metric: BaseMetric) -> tuple[list[str], list[str]]:
    """Extract required and optional parameter names from a metric's ascore method.

    Args:
        metric: A RAGAS collections metric instance.

    Returns:
        Tuple of (required_params, optional_params) where each is a list of
        parameter names.
    """
    sig = inspect.signature(metric.ascore)

    required = []
    optional = []
    for name, param in sig.parameters.items():
        if name == "self":
            continue
        if param.default is inspect.Parameter.empty:
            required.append(name)
        else:
            optional.append(name)

    return required, optional


async def score_with_metric(
    metric: BaseMetric, sample: dict[str, Any]
) -> MetricResult:
    """Score a sample, passing only the parameters the metric's ascore() expects.

    This helper uses introspection to determine which parameters each metric's
    ascore() method requires, then filters the sample dict to only include those
    keys. This allows using a single sample dict format across metrics with
    different signatures.

    Args:
        metric: A RAGAS collections metric instance (e.g., Faithfulness,
            AnswerRelevancy, ContextPrecisionWithReference).
        sample: Dictionary containing evaluation data. Should include all possible
            fields: user_input, response, retrieved_contexts, reference.

    Returns:
        MetricResult from the metric's ascore() method.

    Raises:
        ValueError: If a required parameter is missing from the sample or is None.

    Example:
        >>> from ragas.metrics.collections import Faithfulness, AnswerRelevancy
        >>>
        >>> sample = {
        ...     "user_input": "What is the capital of France?",
        ...     "response": "Paris is the capital of France.",
        ...     "retrieved_contexts": ["Paris is the capital..."],
        ...     "reference": "Paris",  # Only used by some metrics
        ... }
        >>>
        >>> # Works with Faithfulness (needs user_input, response, retrieved_contexts)
        >>> result = await score_with_metric(faithfulness_metric, sample)
        >>>
        >>> # Works with AnswerRelevancy (needs user_input, response)
        >>> result = await score_with_metric(answer_relevancy_metric, sample)
    """
    required, optional = get_ascore_params(metric)

    # Validate required params exist and are not None
    missing = [p for p in required if p not in sample or sample[p] is None]
    if missing:
        raise ValueError(
            f"Metric '{metric.name}' requires the following fields which are "
            f"missing or None: {missing}"
        )

    # Build filtered dict with required + available optional params
    filtered = {k: sample[k] for k in required}
    filtered.update(
        {k: sample[k] for k in optional if k in sample and sample[k] is not None}
    )

    return await metric.ascore(**filtered)


Execute each evaluation metric against each of the samples collected from the traces.  This will generate a score for each metric for each trace.  Send these scores to Langfuse as trace scores.  Print them out in the notebook too.

In [ ]:
for idx, sample in enumerate(samples):
    # evaluate each sample against each of the metrics
    for metric in metrics:
        try:
            result = await score_with_metric(metric, sample)
            langfuse.create_score(
                trace_id=trace_ids[idx],
                name=metric.name,
                value=float(result.value),
                data_type="NUMERIC",
            )
        except Exception as e:
            raise RuntimeError(f"RAGAS evaluation failed: {e}") from e
        print(f"Score for {metric.name}: {result.value} posted to Langfuse!")

# Ensure scores are sent to Langfuse
langfuse.flush()  
